<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261110/%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 機械学習 ― 決定木とランダムフォレスト

本講義では、教師あり学習の代表的な課題である**分類**を主に対象とし、
決定木とランダムフォレストの利用方法を学ぶ。


#機械学習とは

機械学習とは、データから規則性やパターンを学習し、未知のデータに対して
数値や分類の予測を行う方法である。
機械学習では、入力データと結果の例からアルゴリズム（計算）を通じて
**規則性**を学習する。
ここで得られた**規則性**はモデルと呼ばれる。
ただし、機械学習は
**「データを与えれば自動的に適切なモデルが得られる技術」**
ではない。
データの品質、説明変数の選択、評価方法、利用目的などによってモデルの有効性は大きく変化する。

##機械学習の代表的な種類

機械学習は、予測する対象の性質や学習に使用するデータの性質によって
以下のように分類される。

| 種類 | 概要 | 代表的な課題 |
|---|---|---|
| 教師あり学習 | 正解となる目的変数が与えられたデータから関係を学習する | 分類、回帰 |
| 教師なし学習 | 正解となる目的変数を用いず、データの構造や特徴を抽出する | クラスタリング、次元削減 |
| 強化学習 | 行動と報酬の試行錯誤を通じて方策を学習する | 制御、ゲーム、意思決定 |

##分類と回帰

教師あり学習では、目的変数の種類によって主に分類と回帰に分けられる。

- **分類**：目的変数がカテゴリである問題である。例として、良性／悪性、購入する／しない、解約する／しない、などがある。
- **回帰**：目的変数が連続的な数値である問題である。例として、売上金額、需要量、価格、所要時間などの予測がある。

分類の中でも、目的変数が二値のカテゴリカル変数の場合を**二値分類**、
あるいは **2クラス分類**、
3種類以上の場合を**多クラス分類**と呼ぶ。

本講義では**教師あり学習の分類**を扱う。

##機械学習で用いる基本用語

| 用語 | 意味 |
|---|---|
| 説明変数（特徴量） | 予測のために利用する入力データ |
| 目的変数（正解ラベル） | 予測したい対象 |
| モデル | 説明変数から目的変数を予測する仕組み |
| 学習 | データを用いてモデル内部のパラメータを決定する処理 |
| モデルパラメータ | 学習データから自動的に推定される値 |
| ハイパーパラメータ | 学習の前に設定するモデルの構造や学習方法に関する値 |
| 予測 | 学習済みモデルを未知データに適用して結果を出力する処理 |

これから解説する決定木では、
分岐に利用する特徴量や閾値がモデルパラメータに相当する。
一方、木の最大深さ `max_depth` などはハイパーパラメータである。

#分類モデル構築の標準的な流れ

機械学習による分類では、概ね以下の流れでモデルを構築する。

1. **目的を定義する**  
   何を予測し、その予測結果を何に利用するかを明確にする。

2. **データを理解する**  
   項目の意味、データ型、欠損値、外れ値、目的変数の分布などを確認する。

3. **説明変数と目的変数を決める**

4. **学習用データとテストデータに分割する**

5. **前処理を行う**  
   欠損値補完、カテゴリ変数の数値化、必要に応じた標準化などを行う。

6. **モデルを学習する**

7. **テストデータを用いて、未知データに対する性能を評価する**

8. **必要に応じてハイパーパラメータを調整する**

9. **結果を解釈し、利用目的に適しているか判断する**

重要なのは、モデル構築だけでなく、**未知データに対する性能を適切に評価すること**である。

##学習用データとテストデータ

学習に使用したデータだけで性能を評価すると、モデルが学習データを記憶しただけでも高い評価値になる可能性がある。

そこで、データを以下の2つに分け、適切に性能評価が行えるようにする。

- **学習用データ（training data）**：モデルの学習やハイパーパラメータの選択に使用する。
- **テストデータ（test data）**：学習には使用せず、最終的な汎化性能の評価に使用する。

##過学習

モデルが複雑すぎると、学習データには非常によく適合する一方で、未知データに対する予測性能が低下することがある。これを**過学習（overfitting）**
という。
逆に、モデルが単純すぎてデータの構造を十分に表現できない状態を
**underfitting**
という。

機械学習では、学習データへの適合度だけではなく、未知データに対する性能
（これを**汎化性能**と呼ぶ）を高めることが重要である。

#実行環境の準備

以下では、日本語を含むグラフを文字化けせず表示させるために
 `japanize-matplotlib` をインストールする。

In [ ]:
# 日本語表示用ライブラリ
!pip -q install japanize-matplotlib

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import japanize_matplotlib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

#Breast Cancerデータによる分類

##データの読み込み

scikit-learnに含まれる Breast Cancer Wisconsin (Diagnostic) データを利用する。

このデータは569件の観測(レコード数)と
30個の数値特徴量を持つ二値分類データである。
目的変数はがんの診断結果であり、
`cancer.target`に格納されており、
`0 = malignant（悪性）`、`1 = benign（良性）` の2クラス
（2つのカテゴリ値）から構成されている。

In [ ]:
cancer = load_breast_cancer()

df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# 元データのcancer.target: 0=悪性, 1=良性
df["diagnosis"] = cancer.target.astype(int)

print("データ件数・列数:", df.shape)
df.head()

##データの概要確認

まず、欠損値の有無、データ型、目的変数の分布を確認する。

In [ ]:
df.info()

In [ ]:
df["diagnosis"].value_counts().rename(index={0: "悪性（0）", 1: "良性（1）"})

目的変数のクラスが一方に大きく偏っていないかを確認することは重要である。
極端に偏っている場合には、精度の評価に注意が必要である。

##説明変数と目的変数の関係を確認する

例として `mean radius` の分布を、良性・悪性で比較する。

In [ ]:
plot_df = df.copy()
plot_df["diagnosis_label"] = plot_df["diagnosis"].map({0: "悪性", 1: "良性"})

sns.histplot(
    data=plot_df,
    x="mean radius",
    hue="diagnosis_label",
    kde=True,
    element="step"
)
plt.title("mean radius の分布")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

この図から、`mean radius` と診断結果には関係があることが分かる。しかし、分布には重なりがあり、この特徴量だけでは完全に分類はできない。
機械学習モデルでは、複数の説明変数を組み合わせることでこのような重なり部分の少ない分類を目指す。

なお、グラフ上で関連が見られても、それだけで因果関係があるとは判断できない点に注意すること。

##学習用データとテストデータへの分割

まず初めに、
学習用データとテストデータに分割する。
分類問題では、学習用データとテストデータでクラス比率が大きく変わらないように、`stratify=y` を指定する。

In [ ]:
X = df.drop(columns="diagnosis") #説明変数
y = df["diagnosis"] #目的変数

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("全データ:", X.shape)
print("学習用:", X_train.shape)
print("テスト用:", X_test.shape)
print("学習用の悪性比率:", y_train.mean().round(3))
print("テスト用の悪性比率:", y_test.mean().round(3))

#決定木

決定木は、説明変数に対する「Yes / No」の条件分岐を繰り返して予測を行うモデルである。

例えば、

- `worst concave points <= 0.14`
- `worst area <= 900`

のような条件によってデータを順番に分割し、最終的に到達した葉（leaf）でクラスを予測する。

決定木は、予測規則を木構造（分類木と呼ばれる）として可視化できるため、比較的説明しやすいモデルである。

##分岐はどのように決まるか

分類木では、各ノードでクラスができるだけ混ざらないような分割を探す。

`DecisionTreeClassifier` の既定値では、分割の評価に
**ジニ不純度（Gini impurity）**
が用いられる。

あるノードにおいてクラス $k$ の比率を $p_k$ とすると、
ジニ不純度は次式で表される。

$$
G = 1 - \sum_k p_k^2
$$

ノード内がすべて同じクラスであればジニ不純度は0となる。複数クラスが混在するほど値は大きくなる。

決定木は、分割後の不純度が小さくなるように特徴量と閾値を選択する。

##決定木の学習

ここでは木構造を視覚的に確認しやすくするために、まず `max_depth=2` とする。
なお、
決定木では特徴量の大小関係に基づいて分岐するため、通常は標準化を必要としない。

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

tree_model.fit(X_train, y_train)

##決定木の可視化

決定木を可視化する。
図の各ボックスには、分岐条件、ジニ不純度、サンプル数、各クラスの件数、予測クラスなどが表示される。

条件を満たす場合は左側、満たさない場合は右側へ進む。

In [ ]:
plt.figure(figsize=(18, 7))
plot_tree(
    tree_model,
    filled=True,
    feature_names=X.columns,
    class_names=["悪性", "良性"],
    rounded=True,
    fontsize=10
)
plt.title("Breast Cancerデータの決定木（max_depth=2）")
plt.show()

決定木の上位の分岐に利用された特徴量は、
この決定木において分類に有効であった特徴量と考えられる。
ただし、「木の上位に現れた」「特徴量重要度が高い」という事実だけから、その変数が目的変数の原因であると解釈してはならない。また、似た情報を持つ特徴量が複数存在する場合には、どの特徴量が選ばれるかが変化することもある。

##予測確率

分類モデルは最終クラスだけでなく、クラスに対する所属確率を出力できる。

`predict()` は最終的なクラスを返し、`predict_proba()` は各クラスの確率を返す。
なお、ニクラス分類のため良性の確率と悪性の確率の合計が1となっていることを確認せよ。

In [ ]:
proba_example = tree_model.predict_proba(X_test.iloc[:5])

pd.DataFrame(
    proba_example,
    columns=["悪性の確率", "良性の確率"],
    index=X_test.index[:5]
)

#モデル評価と過学習

分類モデルでは、単純な正解率であるAccuracyがよく利用される。しかし、業務上の損失は誤分類の種類によって異なる場合がある。

一般的に、二分類の問題では目的変数のラベルとして
- 1 を付与したものを陽性（Positive）
- 0 を付与したものを陰性（Negative）

と呼び、予測結果を以下の4種類に区別する。

- **True Positive (TP)**：陽性を陽性と予測した
- **False Negative (FN)**：間違って陰性と予測した
- **False Positive (FP)**：間違って陽性と予測した
- **True Negative (TN)**：陰性を陰性と予測した

利用シーンによっては、
FNとFPでは間違うことの影響度が異なることがある。
例えば、健康診断を例にすれば、
病気(悪性)を見逃し健康と予測する、
すなわち**間違って良性**と診断することは重大な誤りとなる。
逆に、迷惑メールの分類では、**間違って悪性**（迷惑）メールと判断することは重大な誤りである。
利用目的に応じて適切な評価指標を選択する必要がある。

##代表的な分類評価指標

| 指標 | 意味 |
|---|---|
| Accuracy | 全予測のうち正解した割合 |
| Precision | 陽性と予測したもののうち、実際に陽性であった割合 |
| Recall | 実際に陽性であるもののうち、陽性と検出できた割合 |
| F1 | PrecisionとRecallの調和平均 |

>調和平均とは、逆数の平均の逆数であり、比率の平均を求める場合に用いられることがある

どの指標を重視するかは利用目的によって異なる。

In [ ]:
y_pred_tree = tree_model.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, y_pred_tree), 3))
print("Precision:", round(precision_score(y_test, y_pred_tree), 3))
print("Recall   :", round(recall_score(y_test, y_pred_tree), 3))
print("F1       :", round(f1_score(y_test, y_pred_tree), 3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_tree,
    cmap="Blues"
)
plt.title("決定木の混同行列")
plt.show()

##木の深さと過学習

決定木では、木を深くすると学習データを細かく分割できるため、学習データに対するAccuracyは高くなりやすい。
しかし、木を深くしすぎると学習データ固有の細かな特徴まで学習してしまい、
未知のデータへの性能（汎か性能）が低下することがある。

以下では `max_depth` を変化させ、
学習用データとテストデータのAccuracyを比較する。

In [ ]:
depth_results = []

for depth in range(1, 16):
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)

    depth_results.append({
        "max_depth": depth,
        "train_accuracy": m.score(X_train, y_train),
        "test_accuracy": m.score(X_test, y_test)
    })

depth_df = pd.DataFrame(depth_results)
depth_df.head()

In [ ]:
depth_df.plot(
    x="max_depth",
    y=["train_accuracy", "test_accuracy"],
    marker="o"
)
plt.ylim(0.7, 1.02)
plt.ylabel("Accuracy")
plt.title("木の深さとAccuracy")
plt.grid(alpha=0.3)
plt.show()

学習データのAccuracyが高いことだけでは、良いモデルとはいえない。
学習データと未知データの性能差が大きい場合は過学習が起きていると考えられる。

#交差検証とハイパーパラメータチューニング

##クロスバリデーション

ハイパーパラメータを選択するためには、学習用データの内部でモデルを比較する必要がある。

そこで用いる方法が
**クロスバリデーション（交差検証）**
である。

k-foldクロスバリデーションでは、学習用データをk個のfoldに分割し、1つを検証用、残りを学習用として評価する操作をk回繰り返す。
分類では、クラス比率をできるだけ維持する **Stratified K-Fold** がよく用いられる。

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    DecisionTreeClassifier(max_depth=2, random_state=42),
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("各foldのAccuracy:", np.round(cv_scores, 3))
print("平均Accuracy:", round(cv_scores.mean(), 3))
print("標準偏差:", round(cv_scores.std(), 3))

##ハイパーパラメータチューニング

決定木には複数のハイパーパラメータがある。

- `max_depth`：木の最大深さ
- `min_samples_split`：ノードを分割するために必要な最小サンプル数
- `min_samples_leaf`：葉に必要な最小サンプル数
- `criterion`：分割の評価基準

最適な値はデータによって異なるため、固定的な「正解値」があるわけではない。

##GridSearchCV

`GridSearchCV` は、指定したハイパーパラメータの組合せを交差検証によって比較する方法である。

ここでは、例として**Recallを最大化**する。
実務では、評価指標は業務目的に応じて決める必要がある。

In [ ]:
#ハイパーパラメータの候補を辞書型のデータとして格納する
param_grid_tree = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_tree,
    scoring="recall",
    cv=cv,
    n_jobs=-1
)

grid_tree.fit(X_train, y_train)

print("最良パラメータ:", grid_tree.best_params_)
print("CVでの最良Recall:", round(grid_tree.best_score_, 3))

`GridSearchCV` は既定で `refit=True` であるため、探索後の `best_estimator_` は、最良条件を用いて学習用データ全体に再学習されたモデルである。

したがって、通常は `best_estimator_` に対して改めて `fit()` を実行する必要はない。

In [ ]:
#最初のホールドアウトしたデータで最終のテストを行う
best_tree = grid_tree.best_estimator_
y_pred_best_tree = best_tree.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, y_pred_best_tree), 3))
print("Precision:", round(precision_score(y_test, y_pred_best_tree), 3))
print("Recall   :", round(recall_score(y_test, y_pred_best_tree), 3))
print("F1       :", round(f1_score(y_test, y_pred_best_tree), 3))

#Titanicデータによる例

ここでは、数値変数とカテゴリ変数、欠損値を含むデータの例としてTitanicデータを利用する。

In [ ]:
titanic = sns.load_dataset("titanic")

print("データ件数・列数:", titanic.shape)
titanic.head()

##主な項目

| 列名 | 意味 |
|---|---|
| survived | 生存フラグ（0=死亡、1=生存） |
| pclass | 乗船クラス（1, 2, 3） |
| sex | 性別 |
| age | 年齢 |
| sibsp | 同乗している兄弟・配偶者の数 |
| parch | 同乗している親・子どもの数 |
| fare | 運賃 |
| embarked | 乗船港 |
| class | pclassに対応するカテゴリ表現 |
| who | sexやage等から作られた派生カテゴリ |
| adult_male | 成人男性かどうかを示す派生項目 |
| deck | デッキ |
| embark_town | embarkedに対応する名称 |
| alive | survivedと同じ結果を文字列で表した項目 |
| alone | sibspやparch等から導かれる派生項目 |

`alive` は目的変数 `survived` と実質的に同じ情報である。


##説明変数と目的変数の設定

本講義ではモデルの説明を単純にするため、以下の基本的な7項目だけを説明変数として利用する。

- 数値変数：`pclass`, `age`, `sibsp`, `parch`, `fare`
- カテゴリ変数：`sex`, `embarked`
- 目的変数：`survived`

最初に学習用データとテストデータに分割し、その後の欠損値補完やOne-Hot Encodingは学習用データから学習する。
この順序はデータリーケージを防ぐために重要である。

In [ ]:
features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

X_titanic = titanic[features]
y_titanic = titanic["survived"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic,
    y_titanic,
    test_size=0.2,
    stratify=y_titanic,
    random_state=42
)

print(X_train_t.isna().sum())

##前処理をPipelineに組み込む

数値変数では中央値による欠損値補完を行い、
カテゴリ変数では最頻値による欠損値補完を行った後、One-Hot Encodingを行ことにする。

これらの前処理を `Pipeline` / `ColumnTransformer` に組み込むことで、交差検証やハイパーパラメータ探索の各foldにおいて、検証データの情報が前処理に混入することを防ぐことができる。

In [ ]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

##決定木モデル

前処理とモデル推定（決定木の作成）を1つのPipelineとして定義し、`.fit()`で推定を実行する。

In [ ]:
titanic_tree = Pipeline([
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42))
])

titanic_tree.fit(X_train_t, y_train_t)

y_pred_titanic_tree = titanic_tree.predict(X_test_t)

print(classification_report(
    y_test_t,
    y_pred_titanic_tree,
    target_names=["死亡", "生存"]
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test_t,
    y_pred_titanic_tree,
    display_labels=["死亡", "生存"],
    cmap="Blues"
)
plt.title("Titanic：決定木の混同行列")
plt.show()

##前処理後の決定木を可視化する

Pipeline内の前処理によって、カテゴリ変数はOne-Hot Encodingされた列に変換されている。

`get_feature_names_out()` を利用すると、変換後の特徴量名を取得できる。

In [ ]:
titanic_tree.named_steps["preprocess"].get_feature_names_out()

In [ ]:
transformed_feature_names = (
    titanic_tree.named_steps["preprocess"].get_feature_names_out()
)

plt.figure(figsize=(18, 8))
plot_tree(
    titanic_tree.named_steps["model"],
    filled=True,
    feature_names=transformed_feature_names,
    class_names=["死亡", "生存"],
    rounded=True,
    fontsize=9
)
plt.title("Titanicデータの決定木")
plt.show()

##Pipeline全体をGridSearchCVで調整する

Pipeline内のモデルのハイパーパラメータは、`ステップ名__パラメータ名` の形式で指定する。

ここでは `model__max_depth` と `model__min_samples_leaf` を探索する。探索範囲をリスト化して、辞書として設定を行う。

In [ ]:
param_grid_titanic_tree = {
    "model__max_depth": [2, 3, 4, 5, None],
    "model__min_samples_leaf": [1, 2, 5, 10]
}

grid_titanic_tree = GridSearchCV(
    estimator=Pipeline([
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    param_grid=param_grid_titanic_tree,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_titanic_tree.fit(X_train_t, y_train_t)

print("最良パラメータ:", grid_titanic_tree.best_params_)
print("CV平均Accuracy:", round(grid_titanic_tree.best_score_, 3))

In [ ]:
best_titanic_tree = grid_titanic_tree.best_estimator_
y_pred_best_titanic_tree = best_titanic_tree.predict(X_test_t)

print("テストAccuracy:",
      round(accuracy_score(y_test_t, y_pred_best_titanic_tree), 3))

#ランダムフォレスト

ランダムフォレストは、多数の決定木を組み合わせて予測を行う**アンサンブル学習**の一種である。
以下のようにパッケージをインポートして利用する。

```
from sklearn.ensemble import RandomForestClassifier
```

単一の決定木では、
学習データの変化によって木構造が大きく変わることがあり、過学習しやすいという弱点を持つ。しかし、
ランダムフォレストでは、主に以下のランダム性を導入して多数の木を学習することで、過学習を抑えることを期待する。

1. 学習データから復元抽出したサンプル（bootstrap sample）を各木の学習に利用する。
2. 各分岐で利用可能な特徴量の一部だけを候補として分岐を探す。
3. 多数の木の予測を統合して最終予測を行う。

分類では、各木の予測を集約することで、単一の決定木より安定した予測が期待できる。

##決定木とランダムフォレストの比較

| 観点 | 決定木 | ランダムフォレスト |
|---|---|---|
| モデル構造 | 1本の木 | 多数の木 |
| 解釈しやすさ | 比較的高い | 全体構造は把握しにくい |
| 過学習 | 起こりやすい | 単一の木より抑えられることが多い |
| 計算量 | 比較的小さい | 木の本数に応じて増える |
| 標準化 | 通常不要 | 通常不要 |
| 主な用途 | 分類・回帰 | 分類・回帰 |

ランダムフォレストでも個々の決定木を可視化することは可能である。しかし、多数の木を統合したモデル全体を1本の決定木のように説明することは難しい。

##Titanicデータでランダムフォレストを学習する

Titanicデータに対する前処理は決定木の場合と同じであるため、
以前に定義した `preprocess` をそのまま利用する。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

titanic_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

titanic_rf.fit(X_train_t, y_train_t)

y_pred_rf = titanic_rf.predict(X_test_t)

print(classification_report(
    y_test_t,
    y_pred_rf,
    target_names=["死亡", "生存"]
))

##ランダムフォレストの主なハイパーパラメータ

以下は、ランダムフォレストでチューニング必要な主要なパラメータである。

| パラメータ | 意味 |
|---|---|
| `n_estimators` | 作成する木の本数 |
| `max_depth` | 各木の最大深さ |
| `min_samples_split` | ノードを分割するために必要な最小サンプル数 |
| `min_samples_leaf` | 葉に必要な最小サンプル数 |
| `max_features` | 各分岐で候補とする特徴量数 |
| `class_weight` | クラスごとの重み |

これらに普遍的な最適値は存在しない。データ量、特徴量の個数、評価指標、計算時間などを考慮して決める必要がある。

候補数が多い場合には、全ての組合せを調べるGrid Searchだけでなく、候補をランダムに探索するRandomized Searchも有効である。

##GridSearchCVによる例

ここでは計算時間を抑えるため、候補を限定して探索する。

In [ ]:
param_grid_rf = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 6, None],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": ["sqrt", None]
}

grid_rf = GridSearchCV(
    estimator=Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ]),
    param_grid=param_grid_rf,
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

grid_rf.fit(X_train_t, y_train_t)

print("最良パラメータ:", grid_rf.best_params_)
print("CV平均Accuracy:", round(grid_rf.best_score_, 3))

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test_t)

print("テストAccuracy:",
      round(accuracy_score(y_test_t, y_pred_best_rf), 3))

#特徴量重要度と解釈上の注意

決定木やランダムフォレストには `feature_importances_` があり、各特徴量が木の分割によって不純度をどの程度減少させたかに基づく重要度を確認できる。

ただし、この重要度には注意が必要である。

- 取り得る値の種類が多い特徴量が高く評価される場合がある。
- 相関の強い特徴量が複数存在すると、重要度が分散することがある。
- 重要度が高いことは因果関係を意味しない。


##Permutation Importance

Permutation Importanceは、特徴量の値をランダムに並べ替えたときに予測性能がどの程度低下するかを測る方法である。
低下の大きな特徴量ほど予測にとって重要なものである、とういう考え方である。
ランダムフォレストだけでなく、線形回帰やニューラルネットワークなど、どんなモデルでも利用可能な評価指標である。

In [ ]:
perm = permutation_importance(
    best_rf,
    X_test_t,
    y_test_t,
    scoring="accuracy",
    n_repeats=20,
    random_state=42
)

perm_df = pd.DataFrame({
    "feature": X_test_t.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df

In [ ]:
perm_plot_df = perm_df.sort_values("importance_mean", ascending=True)

perm_plot_df.plot(
    x="feature",
    y="importance_mean",
    kind="barh",
    legend=False
)
plt.xlabel("Permutation Importance")
plt.ylabel("特徴量")
plt.title("Titanic：Permutation Importance")
#plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

Permutation Importanceも万能ではない。例えば、互いに強く相関した特徴量が存在する場合には、一方を入れ替えても他方が同じ情報を持つため、重要度が小さく見えることがある。

特徴量の解釈では、モデルの仕組みとデータの構造を合わせて考える必要がある。

#モデル比較

同じテストデータを用いて、調整後の決定木とランダムフォレストを比較する。

モデル比較では、テストデータを何度も見ながらモデルを調整しないことが重要である。調整は学習用データ内部の交差検証で行い、テストデータは最終評価に用いる。

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Decision Tree",
        "accuracy": accuracy_score(y_test_t, y_pred_best_titanic_tree),
        "precision": precision_score(y_test_t, y_pred_best_titanic_tree),
        "recall": recall_score(y_test_t, y_pred_best_titanic_tree),
        "f1": f1_score(y_test_t, y_pred_best_titanic_tree)
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test_t, y_pred_best_rf),
        "precision": precision_score(y_test_t, y_pred_best_rf),
        "recall": recall_score(y_test_t, y_pred_best_rf),
        "f1": f1_score(y_test_t, y_pred_best_rf)
    }
])

comparison.round(3)

モデル選択では、次の観点を総合的に判断する必要がある。

- どの種類の誤分類を避けるべきか。
- どの評価指標を重視するか。
- 説明可能性がどの程度必要か。
- 推論速度や計算資源に制約があるか。
- モデルを継続的に監視・更新できるか。
- 利用するデータに倫理・公平性・プライバシー上の問題がないか。

# ランダムフォレストによる回帰（Regression）

ランダムフォレストは分類タスクだけでなく、連続値を予測する**回帰**にも適用可能である。
scikit-learnでは、回帰用モデルとして `RandomForestRegressor` が提供されている。

## 分類と回帰の違い

- **決定木における予測基準**：分類木ではジニ不純度やエントロピーが用いられるが、回帰木では平均二乗誤差（MSE: Mean Squared Error）などが最小化されるように分岐が行われる。
- **予測値の出力**：各決定木の予測値の**平均値**が、ランダムフォレストとしての最終的な予測値となる。

## 実装例

以下では、Google Colabの環境に標準で用意されているカリフォルニアの住宅価格データ（California Housing dataset）を用いて、ランダムフォレスト回帰モデルの構築と評価を行う。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# データの読み込み
train_data = pd.read_csv('/content/sample_data/california_housing_train.csv')
test_data = pd.read_csv('/content/sample_data/california_housing_test.csv')

# 住宅価格（median_house_value）を目的変数に設定する
X_train_reg = train_data.drop(columns='median_house_value')
y_train_reg = train_data['median_house_value']
X_test_reg = test_data.drop(columns='median_house_value')
y_test_reg = test_data['median_house_value']

# モデルの宣言と学習
# 計算時間短縮のため、決定木の数を100、最大深さを10に制限する
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_regressor.fit(X_train_reg, y_train_reg)

# 予測の実行
y_pred_reg = rf_regressor.predict(X_test_reg)

# 評価指標の算出
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R2: {r2:.4f}")

## ハイパーパラメータのチューニングによる精度改善

回帰モデルにおいても、分類モデルと同様にハイパーパラメータを調整することで予測精度を向上させることが可能である。
ここでは `GridSearchCV` を使用し、以下のパラメータ候補から最適な組み合わせを探索する。

- `n_estimators`：作成する決定木の本数（100, 200）
- `max_depth`：各決定木の最大深さ（10, 15, None）
- `min_samples_leaf`：葉ノードに必要な最小サンプル数（1, 5）

In [ ]:
from sklearn.model_selection import GridSearchCV

# チューニング対象のパラメータ候補を設定する
# 探索に時間がかかるので、まずは候補を少なく始める
param_grid_reg = {
    'n_estimators': [100, 200],
#    'max_depth': [10, 15, None],
    'min_samples_leaf': [5]
}

# 5分割の交差検証を用いたグリッドサーチのインスタンスを作成する
# 評価指標には決定係数（R2）を使用する
grid_rf_reg = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid_reg,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

# 学習データを用いて探索を実行する
grid_rf_reg.fit(X_train_reg, y_train_reg)

print("最良パラメータ:", grid_rf_reg.best_params_)
print("CVでの最良R2スコア:", round(grid_rf_reg.best_score_, 4))

## 最適モデルを用いたテストデータの評価

探索によって得られた最良のモデルを用いて、テストデータに対する予測精度を測定し、初期設定モデル（max_depth=10）からの改善状況を確認する。

In [ ]:
# 最良モデルの抽出
best_rf_reg = grid_rf_reg.best_estimator_

# テストデータに対する予測
y_pred_best_reg = best_rf_reg.predict(X_test_reg)

# 評価指標の算出
best_mse = mean_squared_error(y_test_reg, y_pred_best_reg)
best_rmse = np.sqrt(best_mse)
best_r2 = r2_score(y_test_reg, y_pred_best_reg)

print(f"調整後のTest RMSE: {best_rmse:.2f} (初期値: {rmse:.2f})")
print(f"調整後のTest R2: {best_r2:.4f} (初期値: {r2:.4f})")

#まとめ

1. 機械学習では、学習データへの適合ではなく未知データへの汎化性能を評価することが重要である。
2. 決定木は条件分岐を繰り返すことで分類を行い、モデル構造を比較的理解しやすい。
3. 決定木を複雑にしすぎると過学習が起こりやすい。
4. Accuracyだけでなく、Precision、Recall、F1、混同行列を目的に応じて利用する必要がある。
5. ハイパーパラメータの選択には、学習用データでの交差検証を利用する。テストデータでの結果を用いて選択を行ってはならない。
6. 欠損値補完やエンコーディングなどの前処理は、データリーケージを避けるためPipelineに組み込む方法が有効である。
7. ランダムフォレストは多数の決定木を組み合わせることで、単一の決定木より安定した予測が期待できる。
8. 特徴量重要度は有用であるが、因果関係を示すものではなく、計算方法ごとの限界がある。
9. モデル選択では精度だけでなく、利用目的、説明可能性、リスク、運用条件を考慮する必要がある。

# 参考資料

本資料のモデル仕様・実装方法に関する詳細は、scikit-learn公式ドキュメントを参照すること。

- Decision Trees  
  https://scikit-learn.org/stable/modules/tree.html

- DecisionTreeClassifier  
  https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

- RandomForestClassifier  
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

- Cross-validation  
  https://scikit-learn.org/stable/modules/cross_validation.html

- GridSearchCV  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

- Pipeline  
  https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

- Common pitfalls and recommended practices  
  https://scikit-learn.org/stable/common_pitfalls.html

- Permutation feature importance  
  https://scikit-learn.org/stable/modules/permutation_importance.html